# Differential privacy: mechanisms, and what they cost

Three exercises of increasing realism: adding calibrated noise to a single
statistic, to the coefficients of a fitted model, and inside the training loop
itself via `diffprivlib`.

The thread running through all three is that **noise is not error**. The
Laplace and Gaussian mechanisms are unbiased, so averaging enough private
releases recovers the true answer exactly; what privacy costs is variance, and
variance is what the plots below are actually about. Reporting the mean of a
few hundred noisy releases as though it were a result measures the sampling
error of the experiment, not the mechanism.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import trustlab as tl

rng = np.random.default_rng(tl.SEED)

## Part 1 --- the Laplace mechanism on a mean

Five salaries; the query is their mean, and the neighbouring database is the
same five with the first person replaced. To calibrate noise we need a public
bound on what a salary can be: take `[0, 52000]`, which is the range the data
already occupies, treated as a domain constraint rather than as something
inferred from these particular records.

Under bounded DP, replacing one record moves a mean over `n` records by at most
`(hi - lo) / n`. So the two queries have **different** sensitivities --- 10,400
for the five-record mean, 13,000 for the four-record one --- and using the
larger for both, as is easy to do, over-noises the first by 25%.

In [ ]:
with_first = np.array([50000, 52000, 47000, 49000, 51000])
without_first = np.array([52000, 47000, 49000, 51000])

LO, HI = 0.0, 52000.0
sens_with = tl.mean_sensitivity(len(with_first), LO, HI)
sens_without = tl.mean_sensitivity(len(without_first), LO, HI)
true_difference = with_first.mean() - without_first.mean()

print(f"mean with first person     {with_first.mean():>10,.1f}")
print(f"mean without first person  {without_first.mean():>10,.1f}")
print(f"true difference            {true_difference:>10,.1f}")
print(f"sensitivity (n=5)          {sens_with:>10,.1f}")
print(f"sensitivity (n=4)          {sens_without:>10,.1f}")

In [ ]:
EPSILON = 0.5

def noisy_difference(rng):
    a = tl.laplace_mechanism(with_first.mean(), EPSILON, sens_with, rng=rng)
    b = tl.laplace_mechanism(without_first.mean(), EPSILON, sens_without, rng=rng)
    return a - b

print("one release:", f"{noisy_difference(rng):,.1f}",
      f"(true {true_difference:,.1f})")

mean, se, draws = tl.repeat_mean(noisy_difference, trials=200_000, rng=rng)
print(f"\nover 200,000 releases")
print(f"  mean       {mean:>12,.1f}  +/- {se:,.1f} (standard error)")
print(f"  true       {true_difference:>12,.1f}")
print(f"  sd of one  {draws.std(ddof=1):>12,.1f}")
print(f"\nP(|noisy difference| > 10,000) = "
      f"{100 * np.mean(np.abs(draws) > 10_000):.1f}%")

The mean converges on 50 and the standard deviation of a single release is
about 47,000 --- nearly a thousand times the quantity being measured. That
ratio is the result. With only 1,000 draws the standard error is around 1,500,
so a sample mean of a few thousand is an ordinary outcome and says nothing
about bias in the mechanism.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(draws, bins=200, range=(-200_000, 200_000), density=True,
        color="steelblue", alpha=0.8)
ax.axvline(true_difference, color="red", ls="--",
           label=f"true difference ({true_difference:,.0f})")
ax.axvline(0, color="black", lw=0.6)
ax.set_xlabel("noisy difference of means")
ax.set_ylabel("density")
ax.set_title(f"Laplace mechanism, eps = {EPSILON} per query")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

### Sweeping epsilon

Noise scale is `sensitivity / epsilon`, so the spread is inversely proportional
to the budget: a tenfold increase in epsilon is a tenfold narrower histogram
and a tenfold weaker guarantee.

In [ ]:
EPSILONS = [0.1, 0.5, 1.0, 5.0]
TRIALS = 50_000

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
print(f"{'eps':>5} {'noise scale':>13} {'sd of release':>15} {'P(|d|>10k)':>12}")
for ax, eps in zip(axes.ravel(), EPSILONS):
    def draw(rng, eps=eps):
        a = tl.laplace_mechanism(with_first.mean(), eps, sens_with, rng=rng)
        b = tl.laplace_mechanism(without_first.mean(), eps, sens_without, rng=rng)
        return a - b
    _, _, d = tl.repeat_mean(draw, trials=TRIALS, rng=rng)
    print(f"{eps:>5} {sens_without/eps:>13,.0f} {d.std(ddof=1):>15,.0f} "
          f"{100*np.mean(np.abs(d) > 10_000):>11.1f}%")
    ax.hist(d, bins=150, range=(-250_000, 250_000), density=True,
            color="steelblue", alpha=0.8)
    ax.axvline(true_difference, color="red", ls="--")
    ax.set_title(f"eps = {eps}")
    ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

### The Gaussian mechanism

`sigma = sqrt(2 ln(1.25 / delta)) * sensitivity / epsilon` buys
`(epsilon, delta)`-DP instead of pure `epsilon`-DP: a small probability
`delta` of an unbounded leak, in exchange for lighter tails.

At the same nominal epsilon the Gaussian noise here is *wider*, not narrower.
That is not a defect of the mechanism, it is what the relaxation costs when
`delta` is small and the query is one-dimensional --- Gaussian noise pays off
on high-dimensional queries, where the L2 sensitivity grows as `sqrt(k)` rather
than `k`. Note also that the classical bound assumes `epsilon <= 1`, so the
last two rows below are plotted for comparison but carry no guarantee.

In [ ]:
DELTA = 1e-5

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
print(f"{'eps':>5} {'laplace b':>12} {'gaussian sigma':>16} {'guarantee':>12}")
for ax, eps in zip(axes.ravel(), EPSILONS):
    sigma = tl.gaussian_sigma(eps, DELTA, sens_without)
    guarantee = "valid" if eps <= 1 else "eps > 1"
    print(f"{eps:>5} {sens_without/eps:>12,.0f} {sigma:>16,.0f} {guarantee:>12}")

    def draw(rng, eps=eps):
        a = tl.gaussian_mechanism(with_first.mean(), eps, DELTA, sens_with, rng=rng)
        b = tl.gaussian_mechanism(without_first.mean(), eps, DELTA, sens_without, rng=rng)
        return a - b
    _, _, d = tl.repeat_mean(draw, trials=TRIALS, rng=rng)
    ax.hist(d, bins=150, range=(-250_000, 250_000), density=True,
            color="darkorange", alpha=0.8)
    ax.axvline(true_difference, color="red", ls="--")
    ax.set_title(f"eps = {eps}, delta = {DELTA:g}  (sigma = {sigma:,.0f})")
    ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Part 2 --- noising a fitted model

Ordinary least squares on 100 synthetic points with known weights
`[2.0, -3.5]`, then Gaussian noise on the fitted coefficients.

The sensitivity of a regression coefficient to one record is not something you
can assert; it depends on the design matrix and on bounds for the features and
the target, and a linear model with no constraints has unbounded sensitivity in
principle. `COEF_SENSITIVITY` below is a *stipulation* for the sake of the
exercise, and is labelled as one --- output perturbation on OLS is not a
DP-correct recipe without a bound you can actually defend.

Each epsilon is evaluated over many noise draws rather than one, so the curve
shows the mechanism's behaviour instead of a single sample.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

n_samples, n_features = 100, 2
true_weights = np.array([2.0, -3.5])
X = rng.normal(size=(n_samples, n_features))
y = X @ true_weights + rng.normal(scale=2.0, size=n_samples)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3,
                                          random_state=tl.SEED)

reg = LinearRegression().fit(X_tr, y_tr)
base_mse = mean_squared_error(y_te, reg.predict(X_te))
print("fitted weights:", np.round(reg.coef_, 6), " true:", true_weights)
print(f"non-private test MSE: {base_mse:.4f}")

In [ ]:
COEF_SENSITIVITY = 0.05   # stipulated, not derived --- see the note above
DELTA = 1e-5
TRIALS = 500

records = []
for eps in EPSILONS:
    mses = []
    for _ in range(TRIALS):
        noisy = tl.gaussian_mechanism(reg.coef_, eps, DELTA, COEF_SENSITIVITY,
                                      rng=rng, size=reg.coef_.shape)
        mses.append(mean_squared_error(y_te, X_te @ noisy + reg.intercept_))
    mses = np.array(mses)
    records.append((eps, mses.mean(), mses.std(ddof=1) / np.sqrt(TRIALS),
                    np.median(mses)))

print(f"{'eps':>5} {'mean MSE':>12} {'+/- SE':>10} {'median':>12} "
      f"{'vs base':>10}")
for eps, m, se, med in records:
    print(f"{eps:>5} {m:>12.4f} {se:>10.4f} {med:>12.4f} "
          f"{100*(m-base_mse)/base_mse:>9.0f}%")

In [ ]:
eps_arr = np.array([r[0] for r in records])
mean_arr = np.array([r[1] for r in records])
se_arr = np.array([r[2] for r in records])

fig, ax = plt.subplots(figsize=(9, 5))
ax.errorbar(eps_arr, mean_arr, yerr=se_arr, marker="o", capsize=4,
            label=f"private ({TRIALS} draws per point)")
ax.axhline(base_mse, ls="--", c="green", label=f"non-private ({base_mse:.2f})")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("epsilon"); ax.set_ylabel("test MSE")
ax.set_title("privacy budget vs error, averaged over noise draws")
ax.legend(); ax.grid(alpha=0.3, which="both")
plt.tight_layout(); plt.show()

Averaged this way the curve is monotone. A single draw per epsilon is not: the
noise at `eps = 1.0` can easily land further from zero than the noise at
`eps = 0.5`, producing a worse MSE at the weaker privacy setting. That
non-monotonicity is an artefact of the sample size, not a property of the
mechanism, and averaging is the fix rather than a caveat.

## Part 3 --- DP training with diffprivlib

`diffprivlib` puts the noise inside the fit rather than on the output.

The bounds matter here in a way that is easy to skip. If `bounds` and
`data_norm` are not supplied, the library infers them from the data and raises
`PrivacyLeakWarning` --- and it is right to: the bounds are themselves a
function of the private data, so the true privacy cost is larger than the
epsilon you asked for. Supplying them explicitly, from public knowledge of what
the features can be, is what makes the stated epsilon the real one.

In [ ]:
import warnings
import pandas as pd
from diffprivlib.models import LogisticRegression as DPLogisticRegression
from diffprivlib.models import GaussianNB as DPGaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler

df = pd.read_csv(tl.DIABETES_CSV)
X = df.drop(columns=["Outcome"]).values
y = df["Outcome"].values
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3,
                                          random_state=tl.SEED, stratify=y)

scaler = StandardScaler().fit(X_tr)
X_tr_s, X_te_s = scaler.transform(X_tr), scaler.transform(X_te)

clf = LogisticRegression(max_iter=1000).fit(X_tr_s, y_tr)
base_train = clf.score(X_tr_s, y_tr)
base_test = clf.score(X_te_s, y_te)
print(f"non-private logistic regression: train {base_train:.4f}  "
      f"test {base_test:.4f}")

In [ ]:
# Clip to +/- 3 standard deviations after scaling. This is a public decision
# about the feature domain, not a statistic of the data, so it can be stated
# without spending budget -- which is exactly what diffprivlib's
# PrivacyLeakWarning is asking for.
CLIP = 3.0
X_tr_c = np.clip(X_tr_s, -CLIP, CLIP)
X_te_c = np.clip(X_te_s, -CLIP, CLIP)
BOUNDS = (np.full(X_tr_c.shape[1], -CLIP), np.full(X_tr_c.shape[1], CLIP))
DATA_NORM = float(np.sqrt(X_tr_c.shape[1]) * CLIP)   # max L2 norm of a clipped row

epsilons = np.logspace(-2, 2, 50)
REPEATS = 20

train_mean, test_mean, test_sd = [], [], []
with warnings.catch_warnings():
    warnings.simplefilter("error")   # a PrivacyLeakWarning here is a bug, not a note
    for eps in epsilons:
        tr, te = [], []
        for r in range(REPEATS):
            dp = DPLogisticRegression(epsilon=eps, data_norm=DATA_NORM,
                                      random_state=tl.SEED + r)
            dp.fit(X_tr_c, y_tr)
            tr.append(dp.score(X_tr_c, y_tr))
            te.append(dp.score(X_te_c, y_te))
        train_mean.append(np.mean(tr))
        test_mean.append(np.mean(te))
        test_sd.append(np.std(te, ddof=1))

test_mean = np.array(test_mean); test_sd = np.array(test_sd)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
ax.fill_between(epsilons, test_mean - test_sd, test_mean + test_sd,
                alpha=0.25, color="steelblue", label="+/- 1 sd over repeats")
ax.plot(epsilons, test_mean, marker="o", ms=3, color="steelblue",
        label=f"DP test accuracy (mean of {REPEATS})")
ax.plot(epsilons, train_mean, marker="s", ms=3, alpha=0.6, color="grey",
        label=f"DP train accuracy (mean of {REPEATS})")
ax.axhline(base_test, ls="--", c="red", label=f"non-private test ({base_test:.2f})")
ax.axhline(max(np.mean(y_te), 1 - np.mean(y_te)), ls=":", c="black",
           label="majority-class baseline")
ax.set_xscale("log")
ax.set_xlabel("epsilon"); ax.set_ylabel("accuracy")
ax.set_title("DP logistic regression on the Pima diabetes data")
ax.legend(fontsize=9); ax.grid(alpha=0.3, which="both")
plt.tight_layout(); plt.show()

The majority-class line is the one that makes the low-epsilon end readable.
Below about `eps = 0.1` the private model is at or under it, meaning it has
learnt nothing and is simply making noise-driven guesses; single-run curves at
that end swing between 0.35 and 0.65 and neither number is informative.
Averaging over repeats and drawing the band collapses that into what it is.

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris()
Xi_tr, Xi_te, yi_tr, yi_te = train_test_split(iris.data, iris.target,
                                              test_size=0.3,
                                              random_state=tl.SEED,
                                              stratify=iris.target)

nb = GaussianNB().fit(Xi_tr, yi_tr)
print(f"non-private GaussianNB test accuracy: {nb.score(Xi_te, yi_te):.4f}")

# public bounds for iris: the documented measurement range in cm
IRIS_BOUNDS = (np.array([4.0, 2.0, 1.0, 0.1]), np.array([8.0, 4.5, 7.0, 2.5]))

nb_mean, nb_sd = [], []
with warnings.catch_warnings():
    warnings.simplefilter("error")
    for eps in epsilons:
        scores = []
        for r in range(REPEATS):
            dp_nb = DPGaussianNB(epsilon=eps, bounds=IRIS_BOUNDS,
                                 random_state=tl.SEED + r)
            dp_nb.fit(Xi_tr, yi_tr)
            scores.append(dp_nb.score(Xi_te, yi_te))
        nb_mean.append(np.mean(scores))
        nb_sd.append(np.std(scores, ddof=1))

nb_mean = np.array(nb_mean); nb_sd = np.array(nb_sd)

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.fill_between(epsilons, nb_mean - nb_sd, nb_mean + nb_sd, alpha=0.25,
                color="darkorange")
ax.plot(epsilons, nb_mean, marker="o", ms=3, color="darkorange",
        label=f"DP GaussianNB (mean of {REPEATS})")
ax.axhline(nb.score(Xi_te, yi_te), ls="--", c="red", label="non-private")
ax.axhline(1/3, ls=":", c="black", label="chance (3 classes)")
ax.set_xscale("log"); ax.set_xlabel("epsilon"); ax.set_ylabel("accuracy")
ax.set_title("DP Gaussian naive Bayes on iris")
ax.legend(fontsize=9); ax.grid(alpha=0.3, which="both")
plt.tight_layout(); plt.show()

Naive Bayes is noisier than logistic regression at matched epsilon, and the
reason is structural rather than incidental: it perturbs the per-class means
and variances directly, and a noisy *variance* can be driven near zero or
negative, at which point the class-conditional density is nonsense. Logistic
regression noises a gradient that is then averaged over the whole training set,
which absorbs a great deal of it.